# ⏰ Time Series Analysis — Case Study

**Topic:** 07.08 · **Level:** 🟠 Intermediate · **Type:** THEORY + CODE

## 📖 Jhaan-se abhi tak

```python
pd.Timestamp('6th jan 2023 8:10')   # single moment
pd.DatetimeIndex(...)                # collection
```

- Recap: Timestamp, DatetimeIndex, date_range, to_datetime, dt accessor
- Ab: Timedelta, Resampling, Rolling, Shifting

In [89]:
import pandas as pd
import numpy as np

In [101]:
# till now
# Timestamp
pd.Timestamp('6th jan 2023 8:10')
# DatetimeIndex -> df and series index
pd.DatetimeIndex([pd.Timestamp('6th jan 2023 8:10'),pd.Timestamp('7th jan 2023 8:10'),pd.Timestamp('8th jan 2023 8:10')])[0]
# date_range()
pd.date_range(start='2023-1-6',end='2023-1-31',freq='D')
# to_datetime()
s = pd.Series(['2023/1/6','2023/1/7','2023/1/7'])
pd.to_datetime(s).dt.day_name()

0      Friday
1    Saturday
2    Saturday
dtype: object

### Timedelta Object

Represents a duration, the difference between two dates or times.

## 📖 Timedelta — duration

```python
t1 = pd.Timestamp('6th Jan 2023 8:20:14')
t2 = pd.Timestamp('7th Jan 2023 10:00:00')
t2 - t1                       # Timedelta object
pd.Timedelta(days=2, hours=10, minutes=35)
```

- Timedelta = 2 dates/times **ka difference** (duration)
- Ya standalone `pd.Timedelta(days, hours, ...)`
- Arithmetic: `Timestamp + Timedelta` → naya Timestamp
- `date_range - Timedelta` → shifted dates

## 🔬 Deep Dive : Timedelta — duration between dates

```python
t1 = pd.Timestamp('6th Jan 2023 8:20:14'); t2 = pd.Timestamp('9th Jan 2023 6:00:00')
t2 - t1                      # → Timedelta 2 days 21:39:46
pd.Timedelta(days=2, hours=10, minutes=35)   # standalone duration
pd.Timestamp('6th jan 2023') + pd.Timedelta(days=2, hours=10, minutes=35)

df['delivery_time_period'] = df['delivery_date'] - df['order_date']
df['delivery_time_period'].dt.days    # integer days
```

- Timedelta = DURATION (difference), Timestamp = a MOMENT
- subtract Timestamps → Timedelta
- Arithmetic with Timedelta shifts dates
- .dt.days / .total_seconds() — extract duration size
- Real: order→delivery time = delivery_date - order_date column
- Rollowing: avg delivery time, late order detection
- As dtype: timedelta64[ns] — vector-friendly
- Compare durations directly
- Useful for SLA/completion time metrics

In [105]:
# create using Timestamp objects
t1 = pd.Timestamp('6th Jan 2023 8:20:14')
t2 = pd.Timestamp('26th Jan 2023 10:00:00')

t2 - t1

Timedelta('20 days 01:39:46')

In [108]:
# standalone creation
pd.Timedelta(days=2,hours=10,minutes=35)

Timedelta('2 days 10:35:00')

In [109]:
# Arithmetic
pd.Timestamp('6th jan 2023') + pd.Timedelta(days=2,hours=10,minutes=35)

Timestamp('2023-01-08 10:35:00')

In [112]:
pd.date_range(start='2023-1-6',end='2023-1-31',freq='D') - pd.Timedelta(days=2,hours=10,minutes=35)

DatetimeIndex(['2023-01-03 13:25:00', '2023-01-04 13:25:00',
               '2023-01-05 13:25:00', '2023-01-06 13:25:00',
               '2023-01-07 13:25:00', '2023-01-08 13:25:00',
               '2023-01-09 13:25:00', '2023-01-10 13:25:00',
               '2023-01-11 13:25:00', '2023-01-12 13:25:00',
               '2023-01-13 13:25:00', '2023-01-14 13:25:00',
               '2023-01-15 13:25:00', '2023-01-16 13:25:00',
               '2023-01-17 13:25:00', '2023-01-18 13:25:00',
               '2023-01-19 13:25:00', '2023-01-20 13:25:00',
               '2023-01-21 13:25:00', '2023-01-22 13:25:00',
               '2023-01-23 13:25:00', '2023-01-24 13:25:00',
               '2023-01-25 13:25:00', '2023-01-26 13:25:00',
               '2023-01-27 13:25:00', '2023-01-28 13:25:00'],
              dtype='datetime64[ns]', freq='D')

In [113]:
# real life example
df = pd.read_csv('deliveries.csv')
df.head()

  order_date delivery_date
0    5/24/98        2/5/99
1    4/22/92        3/6/98
2    2/10/91       8/26/92
3    7/21/92      11/20/97
4     9/2/93       6/10/98

In [117]:
df['order_date'] = pd.to_datetime(df['order_date'])
df['delivery_date'] = pd.to_datetime(df['delivery_date'])

## 📖 Real-life — delivery time

```python
df['delivery_date'] - df['order_date']    # duration per row
df['delivery_time_period'] = ...
```

- Difference of 2 datetime cols → Timedelta col
- Business metric — delivery time
- `.dt.days` se extract karo

In [121]:
df['delivery_time_period'] = df['delivery_date'] - df['order_date']

df['delivery_time_period'].mean()

Timedelta('1217 days 22:53:53.532934128')

### Time series

A time series is a data set that tracks a sample over time. In particular, a time series allows one to see what factors influence certain variables from period to period. Time series analysis can be useful to see how a given asset, security, or economic variable changes over time.

Examples

- Financial Data (Company stocks)
- Natural Data (Rainfall measurement)
- Event Data (Covid)
- Medical Data (Heart rate monitoring)

Types of Operations done on Time Series

- Time Series Analysis
- Time Series Forecasting

## 📖 Time series data

```python
google = pd.read_csv('google.csv')
google['Date'] = pd.to_datetime(google['Date'])
google.set_index('Date', inplace=True)
```

- Time series = time pe order kiya gya data
- Datetime → index banake datetime elegance
- `loc['2021-12-30']` — specific date
- Partial indexing: `yt.loc['2022']`, `yt.loc['2022-12']`
- `.index.month_name()` etc — labels

In [122]:
google = pd.read_csv('google.csv')
google.head()

         Date       Open       High        Low      Close  Adj Close    Volume
0  2004-08-19  49.813290  51.835709  47.800831  49.982655  49.982655  44871361
1  2004-08-20  50.316402  54.336334  50.062355  53.952770  53.952770  22942874
2  2004-08-23  55.168217  56.528118  54.321388  54.495735  54.495735  18342897
3  2004-08-24  55.412300  55.591629  51.591621  52.239197  52.239197  15319808
4  2004-08-25  52.284027  53.798351  51.746044  52.802086  52.802086   9232276

In [123]:
google.tail()

            Date         Open         High          Low        Close  \
4466  2022-05-16  2307.679932  2332.149902  2286.699951  2295.850098   
4467  2022-05-17  2344.550049  2344.550049  2306.750000  2334.030029   
4468  2022-05-18  2304.750000  2313.913086  2242.840088  2248.020020   
4469  2022-05-19  2236.820068  2271.750000  2209.360107  2214.909912   
4470  2022-05-20  2241.709961  2251.000000  2127.459961  2186.260010   

        Adj Close   Volume  
4466  2295.850098  1164100  
4467  2334.030029  1078800  
4468  2248.020020  1399100  
4469  2214.909912  1459600  
4470  2186.260010  1878100  

In [124]:
subs = pd.read_csv('subscribers.csv')
views = pd.read_csv('views.csv')
wt = pd.read_csv('watch-time.csv')
comments = pd.read_csv('comments.csv')

In [131]:
comments.tail()

            Date  Comments added
1354  2022-12-31              20
1355  2023-01-01              13
1356  2023-01-02              66
1357  2023-01-03              22
1358  2023-01-04              21

In [135]:
yt = subs.merge(views,on='Date').merge(wt,on='Date').merge(comments,on='Date')

In [136]:
yt.head()

         Date  Subscribers  Views  Watch time (hours)  Comments added
0  2019-04-17            0      0                 0.0               0
1  2019-04-18            0      0                 0.0               0
2  2019-04-19            0      0                 0.0               0
3  2019-04-20            0      0                 0.0               0
4  2019-04-21            0      0                 0.0               0

In [139]:
google['Date'] = pd.to_datetime(google['Date'])
yt['Date'] = pd.to_datetime(yt['Date'])

In [141]:
yt.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 1359 entries, 0 to 1358
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Date                1359 non-null   datetime64[ns]
 1   Subscribers         1359 non-null   int64         
 2   Views               1359 non-null   int64         
 3   Watch time (hours)  1359 non-null   float64       
 4   Comments added      1359 non-null   int64         
dtypes: datetime64[ns](1), float64(1), int64(3)
memory usage: 63.7 KB


In [143]:
google.set_index('Date',inplace=True)
yt.set_index('Date',inplace=True)

In [144]:
yt.head()

            Subscribers  Views  Watch time (hours)  Comments added
Date                                                              
2019-04-17            0      0                 0.0               0
2019-04-18            0      0                 0.0               0
2019-04-19            0      0                 0.0               0
2019-04-20            0      0                 0.0               0
2019-04-21            0      0                 0.0               0

## 🔬 Deep Dive : DatetimeIndex — partial indexing & slicing

```python
google.set_index('Date', inplace=True)
google.loc['2021-12-30']        # exact date
yt.loc['2022-12']               # PARTIAL: whole month
yt.loc['2022']                  # whole year
yt.loc['2022-12-15':'2023-1-1':2]   # date-range slice with step
yt.index.month_name()           # level 'month_name' derived
```

- Setting Date as index unlocks date-powered lookup
- Partial index ('2022-12', '2022') — pandas figures range automatically
- .loc slices inclusive on dates like labels
- derived from index: .index.month_name()/.day_name()
- challenges: exact-date-every-year needs day/month extraction
- Frequency accessories: index.year, index.quarter
- Fast range queries with sorted index
- Chronological plotting axis labels
- Principle: set meaningful index early

In [148]:
# fetch a specific date
google.loc['2021-12-30']

Open           2929.000000
High           2941.250000
Low            2915.169922
Close          2920.050049
Adj Close      2920.050049
Volume       648900.000000
Name: 2021-12-30 00:00:00, dtype: float64

In [150]:
# partial indexing -> select a particular year/month
yt.loc['2022-12']
yt.loc['2022']

            Subscribers  Views  Watch time (hours)  Comments added
Date                                                              
2022-01-01           30   3111            327.3357              11
2022-01-02           42   4109            402.5877               5
2022-01-03           45   4264            431.8496              13
2022-01-04           53   4160            427.6701               7
2022-01-05           32   3971            433.6201               8
...                 ...    ...                 ...             ...
2022-12-27          201  18245           2228.4891              18
2022-12-28          191  18355           2174.0347              21
2022-12-29          142  17247           2115.5851              21
2022-12-30          157  15044           1828.5584              10
2022-12-31          116  12468           1446.5901              20

[365 rows x 4 columns]

In [158]:
# create year month-name and day-name
yt['month_name'] = yt.index.month_name()
yt['weekday_name'] = yt.index.day_name()
yt['quarter'] = yt.index.quarter

In [159]:
yt.head()

            Subscribers  Views  Watch time (hours)  Comments added month_name  \
Date                                                                            
2019-04-17            0      0                 0.0               0      April   
2019-04-18            0      0                 0.0               0      April   
2019-04-19            0      0                 0.0               0      April   
2019-04-20            0      0                 0.0               0      April   
2019-04-21            0      0                 0.0               0      April   

           weekdday_name  quarter weekday_name  
Date                                            
2019-04-17         April        2    Wednesday  
2019-04-18         April        2     Thursday  
2019-04-19         April        2       Friday  
2019-04-20         April        2     Saturday  
2019-04-21         April        2       Sunday  

In [160]:
google['month_name'] = google.index.month_name()
google['weekday_name'] = google.index.day_name()
google['quarter'] = google.index.quarter

google.head()

                 Open       High        Low      Close  Adj Close    Volume  \
Date                                                                          
2004-08-19  49.813290  51.835709  47.800831  49.982655  49.982655  44871361   
2004-08-20  50.316402  54.336334  50.062355  53.952770  53.952770  22942874   
2004-08-23  55.168217  56.528118  54.321388  54.495735  54.495735  18342897   
2004-08-24  55.412300  55.591629  51.591621  52.239197  52.239197  15319808   
2004-08-25  52.284027  53.798351  51.746044  52.802086  52.802086   9232276   

           month_name weekday_name  quarter  
Date                                         
2004-08-19     August     Thursday        3  
2004-08-20     August       Friday        3  
2004-08-23     August       Monday        3  
2004-08-24     August      Tuesday        3  
2004-08-25     August    Wednesday        3  

In [162]:
# slicing
yt.loc['2022-12-15':'2023-1-1':2]

            Subscribers  Views  Watch time (hours)  Comments added month_name  \
Date                                                                            
2022-12-15          176  20396           2730.0469              24   December   
2022-12-17          180  16207           1967.5637              20   December   
2022-12-19          171  19193           2251.5935              30   December   
2022-12-21          185  20380           2781.9778              25   December   
2022-12-23          144  15855           1882.2075              22   December   
2022-12-25          148  16026           2033.8560              32   December   
2022-12-27          201  18245           2228.4891              18   December   
2022-12-29          142  17247           2115.5851              21   December   
2022-12-31          116  12468           1446.5901              20   December   

           weekdday_name  quarter weekday_name  
Date                                            
2022-12-15

In [170]:
# challenge -> fetch info for a particular date every year -> limitation of timedelta
google.head()


google[google.index.isin(pd.date_range(start='2005-1-6',end='2022-1-6',freq=pd.DateOffset(years=1)))]

                   Open         High          Low        Close    Adj Close  \
Date                                                                          
2005-01-06    97.175758    97.584229    93.509506    93.922951    93.922951   
2006-01-06   227.581970   234.371521   225.773743   231.960556   231.960556   
2009-01-06   165.868286   169.763687   162.585587   166.406265   166.406265   
2010-01-06   311.761444   311.761444   302.047852   302.994293   302.994293   
2011-01-06   304.199799   308.060303   303.885956   305.604523   305.604523   
2012-01-06   328.344299   328.767700   323.681763   323.796326   323.796326   
2014-01-06   554.426880   557.340942   551.154114   556.573853   556.573853   
2015-01-06   513.589966   514.761719   499.678131   500.585632   500.585632   
2016-01-06   730.000000   747.179993   728.919983   743.619995   743.619995   
2017-01-06   795.260010   807.900024   792.203979   806.150024   806.150024   
2020-01-06  1350.000000  1396.500000  1350.000000  1

## 📖 Visualization

```python
google['Close'].plot()
yt.groupby('month_name')['Subscribers'].mean().plot(kind='bar')
yt.groupby('quarter')['Close'].mean().plot(kind='bar')
```

- Datetime index → plot automatically sorted x-axis
- Monthly/quarterly trend — groupby + mean
- Yearly/month wise patterns dekhne me hep

## 🔬 Deep Dive : Time series visualization

```python
google['Close'].plot()                       # line over time
yt['Subscribers'].plot()
yt.loc['2022-12']['Subscribers'].plot()     # single month window
yt.groupby('month_name')['Subscribers'].mean().plot(kind='bar')
google.groupby('quarter')['Close'].mean().plot(kind='bar')
```

- After datetime index → .plot() time axis automatic
- Single col / subset window plots
- groupby month_name/year/quarter → trend bars
- Aggregation before plotting = smoother trends
- Multiple cols: yt[['Subscribers','Views',...]].plot()
- Layout: df.plot(subplots=True, layout=(3,2), sharex=False)
- Longer range zoom via loc slices
- Visual pattern: each trends, seasonality, spikes
- Bar for categorical time, line for continuous
- default figure size adjustable

In [172]:
# viz a single col
google['Close'].plot()

<Figure size 432x288 with 1 Axes>

In [173]:
yt['Subscribers'].plot()

<Figure size 432x288 with 1 Axes>

In [176]:
# viz all cols together
subset_yt = yt[['Subscribers', 'Views', 'Watch time (hours)', 'Comments added']]
ax = subset_yt.plot(subplots=True,
                    layout=(2, 2),
                     sharex=False,
                     sharey=False,
                     linewidth=0.7,
                     fontsize=10,
                     legend=False,
                      figsize=(20,10),
                      title=['Subscribers', 'Views', 'Watch time (hours)', 'Comments'])

<Figure size 1440x720 with 4 Axes>

In [182]:
# plot for a particular year/month/week
yt.loc['2022-12']['Subscribers'].plot()

<Figure size 432x288 with 1 Axes>

In [181]:
google.loc['2021-12']['Close'].plot()

<Figure size 432x288 with 1 Axes>

In [209]:
# monthly trend
yt.groupby('month_name')['Subscribers'].mean().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [210]:
# yearly trend
yt.groupby('weekday_name')['Subscribers'].mean().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [212]:
google.groupby('month_name')['Close'].mean().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [213]:
# quaterly trend
google.groupby('quarter')['Close'].mean().plot(kind='bar')

<Figure size 432x288 with 1 Axes>

In [189]:
# frequency
google.index

DatetimeIndex(['2004-08-19', '2004-08-20', '2004-08-23', '2004-08-24',
               '2004-08-25', '2004-08-26', '2004-08-27', '2004-08-30',
               '2004-08-31', '2004-09-01',
               ...
               '2022-05-09', '2022-05-10', '2022-05-11', '2022-05-12',
               '2022-05-13', '2022-05-16', '2022-05-17', '2022-05-18',
               '2022-05-19', '2022-05-20'],
              dtype='datetime64[ns]', name='Date', length=4632, freq='B')

## 📖 Frequency & asfreq

```python
google.asfreq('6H', method='bfill')
```

- `asfreq(freq)` — index ko spec frequency pe
- Missing points — `method='ffill'/'bfill'` fill
- Upsample/downsample alignment

## 🔬 Deep Dive : Frequency & asfreq

```python
google.index            # check current frequency (eg 'D')
google.asfreq('6H')                     # upsample frequent Higher freq
google.asfreq('6H', method='bfill')     # fill missing by backfill
```

- freq = time spacing between observations
- asfreq — CONVERT index frequency (no aggregation)
- higher freq (D→6H) = upsample — gaps created → filling method needed
- lowest freq (6H→D) = down
- method: bfill (back fill — next), ffill (forward — previous)
- Creates regular grid for missing
- Compare same date across time enabling
- Aligns irregular data to uniform steps
- asfreq vs resample — asfreq layout only; resample does aggregate
- Padding = where values appear duplicated continuous assumption

In [196]:
# asfreq
google.asfreq('6H',method='bfill')

                            Open         High          Low        Close  \
Date                                                                      
2004-08-19 00:00:00    49.813290    51.835709    47.800831    49.982655   
2004-08-19 06:00:00    50.316402    54.336334    50.062355    53.952770   
2004-08-19 12:00:00    50.316402    54.336334    50.062355    53.952770   
2004-08-19 18:00:00    50.316402    54.336334    50.062355    53.952770   
2004-08-20 00:00:00    50.316402    54.336334    50.062355    53.952770   
...                          ...          ...          ...          ...   
2022-05-19 00:00:00  2236.820068  2271.750000  2209.360107  2214.909912   
2022-05-19 06:00:00  2241.709961  2251.000000  2127.459961  2186.260010   
2022-05-19 12:00:00  2241.709961  2251.000000  2127.459961  2186.260010   
2022-05-19 18:00:00  2241.709961  2251.000000  2127.459961  2186.260010   
2022-05-20 00:00:00  2241.709961  2251.000000  2127.459961  2186.260010   

                       A

### Resampling

Resampling involves changing the frequency of your time series observations.

Two types of resampling are:

**Upsampling**: Where you increase the frequency of the samples, such as from minutes to seconds.<br>
**Downsampling**: Where you decrease the frequency of the samples, such as from days to months.

## 📖 Resampling

```python
yt['Subscribers'].resample('Y').mean()    # down — aggregate
google['Close'].resample('12H').interpolate(method='spline', order=2)  # up
```

- **Downsampling** — higher freq → lower (sum/mean)
- **Upsampling** — lower → higher (interpolate)
- `resample('Y')` daily → yearly
- `interpolate` — gaps smooth fill

## 🔬 Deep Dive : Resampling

```python
yt['Subscribers'].resample('Y').mean()      # DOWNSAMPLE → YEAR aggregate
google['Close'].resample('12H')             # UPSAMPLE → 12-hour grid
yt['Subscribers'].resample('M').sum()
# downsampling: plot smoother monthly/yearly trend
# upsampling: missing points via interpolate()
google['Close'].resample('12H').interpolate(method='spline', order=2)
```

- resample = groupby for time — CHANGE FREQUENCY + AGGREGATE
- Downsample (high→low freq): mean/sum/max per period (smooth trend)
- Upsample (low→high freq): new points → interpolate fills
  - linear, spline(order), ffill/bfill
- Rules: D/M/Y/Q/H; 'M' month-end, 'Y' year-end
- Aggregations per period like groupby
- Patterns: quarterly report, daily → weekly sales
- Rolling mean equivalent but period based
- Requires datetime index (sorted)
- Contrast: asfreq layout only; resample + aggregate

In [222]:
# Downsampling
yt['Subscribers'].plot()
yt['Subscribers'].resample('Y').mean().plot()

<Figure size 432x288 with 1 Axes>

In [229]:
# Upsampling
google['Close'].resample('12H').interpolate(method='spline',order=2).plot()

<Figure size 432x288 with 1 Axes>

### Rolling Window(Smoothing)
Time series data in original format can be quite volatile, especially on smaller aggregation levels. The concept of rolling, or moving averages is a useful technique for smoothing time series data. 

## 📖 Rolling window (smoothing)

```python
yt['Subscribers'].rolling(30).mean()   # 30-day avg
```

- Window itna data ka mean nikalte jao
- Volatile series smoothen karne ke liye
- Trend ko visible banata hai

## 🔬 Deep Dive : Rolling window — smoothing

```python
yt['Subscribers'].rolling(30).mean()            # 30-day MOVING average
yt['Subscribers'].rolling(30).mean().plot(title='rolling')
yt['Subscribers'].rolling(30).max() / .std()
```

- rolling(n) — rolling WINDOW of last n observations then agg
- rolling mean = moving average — SMOOTHS noise, shows trend
- Volatile daily → clean mAvg curve
- window= n (rows) OR offset ('30D' time-based)
- agg: mean/max/std/min/sum
- Rolling std — volatility measure
- First n-1 → NaN (small window initials)
- The classic time-series smoothing tool
- vs resample: rolling = moving window; resample = buckets periods

In [244]:
# Rolling window
yt['Subscribers'].rolling(30).mean().plot(title='rolling')
yt['Subscribers'].ewm(30).mean().plot(title='ewm')


<Figure size 432x288 with 1 Axes>

### Shifting

The shift() function is Pandas is used to, well, shift the entire series up or down by the desired number of periods.

## 📖 Shift

```python
yt['Subscribers'].shift(1)   # ek step peeche
```

- `shift(n)` — values n positions shift
- Lag features (prev day values)
- Login analysis: prev login time minus current
- Feature engineering ka tool

## 🔬 Deep Dive : Shift — time lag

```python
yt['Subscribers'].shift(1)      # PREVIOUS value at current row
yt['Subscribers'].shift(-1)     # NEXT value (negative)
user_df['shifted'] = user_df['login_time'].shift(1)
(user_df['login_time'] - user_df['shifted'])   # time BETWEEN logins
```

- shift(k) — move values k steps (creates LAG column)
- shift(1) = previous value; shift(-1) = next value
- First row shift(1) → NaN (nothing before)
- Compute differences: current - shifted = per-step change
- Login time analysis: gap between consecutive logins
- Feature engineering: yesterday's value as predictor
- % change, momentum detections
- Align by position, not date
- lag features = forecasting bread & butter
- You can shift per group too

In [247]:
# shift
yt['Subscribers']

Date
2019-04-17      0
2019-04-18      0
2019-04-19      0
2019-04-20      0
2019-04-21      0
             ... 
2022-12-31    116
2023-01-01    142
2023-01-02    171
2023-01-03    162
2023-01-04    147
Name: Subscribers, Length: 1359, dtype: int64

In [250]:
yt['Subscribers'].shift(-1)

Date
2019-04-17      0.0
2019-04-18      0.0
2019-04-19      0.0
2019-04-20      0.0
2019-04-21      0.0
              ...  
2022-12-31    142.0
2023-01-01    171.0
2023-01-02    162.0
2023-01-03    147.0
2023-01-04      NaN
Name: Subscribers, Length: 1359, dtype: float64

In [256]:
# shift example
df = pd.read_csv('/content/login.csv',header=None)
df = df[[1,2]]
df.head()
df.rename(columns={1:'user_id',2:'login_time'},inplace=True)
df.head()

   user_id           login_time
0      466  2017-01-07 18:24:07
1      466  2017-01-07 18:24:55
2      458  2017-01-07 18:25:18
3      458  2017-01-07 18:26:21
4      592  2017-01-07 19:09:59

In [268]:
user_df = df[df['user_id'] == 458]
user_df.head()

    user_id           login_time
2       458  2017-01-07 18:25:18
3       458  2017-01-07 18:26:21
9       458  2017-01-09 11:13:12
10      458  2017-01-09 11:34:02
25      458  2017-01-10 12:14:11

In [269]:
user_df['login_time'] = pd.to_datetime(user_df['login_time'])
user_df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 208 entries, 2 to 1018
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   user_id     208 non-null    int64         
 1   login_time  208 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1)
memory usage: 4.9 KB


<ipython-input-269-fd41e73e6ce6>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_df['login_time'] = pd.to_datetime(user_df['login_time'])


In [270]:
user_df['shifted'] = user_df['login_time'].shift(1)
(user_df['login_time'] - user_df['shifted']).mean()

<ipython-input-270-091b95be4a6b>:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  user_df['shifted'] = user_df['login_time'].shift(1)


Timedelta('0 days 17:29:22.053140096')

In [ ]:
ax = df.plot(subplots=True,
                    layout=(3, 2),
                     sharex=False,
                     sharey=False,
                     linewidth=0.7,
                     fontsize=10,
                     legend=False,
                      figsize=(20,15),
                      title=['Open', 'High', 'Low', 'Close','Adjusted Close', 'Volume'])